# Group 5: Exercises

This notebook contains solutions to exercise 5 from Chapter 5 of "An Introduction to Statistical Learning" (Python edition).

## Exercise 5: Validation Set Approach on Default Data

We apply logistic regression to predict `default` using `income` and `balance`.

We estimate the test error using the validation set approach and repeat it with three different splits.

## 1. Imports and Data Loading

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import summarize
from sklearn.model_selection import train_test_split

# Load the Default dataset
Default = load_data('Default')
Default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


## (a) Fit Logistic Regression Model
We fit a logistic regression (using statsmodels GLM with Binomial family) predicting default using income and balance.

In [2]:
# Define the model formula: default ~ income + balance
X = Default[['income', 'balance']]
X = sm.add_constant(X)   # add intercept
y = Default['default'].apply(lambda x: 1 if x == 'Yes' else 0)   # binary response

# Fit using GLM with Binomial family
model = sm.GLM(y, X, family=sm.families.Binomial())
results = model.fit()
print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                default   No. Observations:                10000
Model:                            GLM   Df Residuals:                     9997
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -789.48
Date:                Thu, 27 Aug 2026   Deviance:                       1579.0
Time:                        13:52:01   Pearson chi2:                 6.95e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1256
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -11.5405      0.435    -26.544      0.0

## (b) Validation Set Approach – Single Split

We split the data into a training set (50%) and a validation set (50%).

We fit the model on the training set, predict probabilities for the validation set, classify as default if probability > 0.5, and compute the misclassification error.

In [3]:
# Set seed for reproducibility
np.random.seed(0)

# Split 50/50
train, valid = train_test_split(Default, test_size=0.5, random_state=0)

# Prepare training data
X_train = sm.add_constant(train[['income', 'balance']])
y_train = (train['default'] == 'Yes').astype(int)

# Fit logistic regression
model_train = sm.GLM(y_train, X_train, family=sm.families.Binomial())
res_train = model_train.fit()

# Predict on validation set
X_valid = sm.add_constant(valid[['income', 'balance']])
prob = res_train.predict(X_valid)
pred = (prob > 0.5).astype(int)
true = (valid['default'] == 'Yes').astype(int)

# Compute misclassification error
error = np.mean(pred != true)
print(f"Validation error (split 1): {error:.4f}")

Validation error (split 1): 0.0290


## (c) Repeat with Three Different Splits
We now perform the same procedure three times, each with a different random seed, and compare the results.

In [4]:
seeds = [1, 2, 3]
errors = []

for seed in seeds:
    np.random.seed(seed)
    train, valid = train_test_split(Default, test_size=0.5, random_state=seed)
    
    X_train = sm.add_constant(train[['income', 'balance']])
    y_train = (train['default'] == 'Yes').astype(int)
    model_train = sm.GLM(y_train, X_train, family=sm.families.Binomial())
    res_train = model_train.fit()
    
    X_valid = sm.add_constant(valid[['income', 'balance']])
    prob = res_train.predict(X_valid)
    pred = (prob > 0.5).astype(int)
    true = (valid['default'] == 'Yes').astype(int)
    
    err = np.mean(pred != true)
    errors.append(err)
    print(f"Seed {seed}: validation error = {err:.4f}")

print("\nAverage error over three splits:", np.mean(errors))

Seed 1: validation error = 0.0250
Seed 2: validation error = 0.0248
Seed 3: validation error = 0.0248

Average error over three splits: 0.024866666666666665


*We observe that the validation error varies somewhat across splits, but remains relatively low (around 2.5%).<br>
The model does a good job predicting default using only income and balance.*

## (d) Further Comment
Because default is highly imbalanced (only ~3% of observations are defaults), the misclassification rate is dominated by correct classification of non‑defaults.

The model still has room for improvement, but the validation set approach gives a reasonable estimate of test error.

However, the results are sensitive to the particular split; cross‑validation would provide a more stable estimate.